In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

In [ ]:
import os
import sys

# 1. Manually add the CUDA and cuDNN paths to the system environment
# This points TensorFlow directly to the drivers you just installed
env_path = os.path.join(sys.prefix, 'Library', 'bin')
os.environ['PATH'] = env_path + os.pathsep + os.environ['PATH']

import tensorflow as tf
import numpy as np

print("--- Hardware Check ---")
# 2. Check again now that the path is forced
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ SUCCESS: RTX 4050 Detected! {gpus}")
else:
    print("❌ GPU still missing. We need to check the PATH variables.")

print(f"\nTensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")

In [ ]:
import cv2
import os

def extract_frames(video_path):
    # Create a folder to store the frames
    output_folder = 'project_frames'

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Open the video file
    vidcap = cv2.VideoCapture(video_path)
    success, image = vidcap.read()
    count = 0
    frame_id = 0

    print(f"Starting extraction from: {video_path}")
    
    while success:
        # We save 1 frame every 30 frames (approx. 1 frame per second of video)
        if count % 30 == 0:
            cv2.imwrite(f"{output_folder}/frame_{frame_id}.jpg", image)     
            frame_id += 1
        success, image = vidcap.read()
        count += 1
    
    vidcap.release()
    print(f"✅ Done! {frame_id} frames saved in '{output_folder}' folder.")

# To run this, uncomment the line below and change 'test.mp4' to your file name:
# extract_frames('test.mp4')

In [ ]:
from tensorflow.keras.applications.inception_resnet_v2 import InceptionResNetV2, preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the model with 'ImageNet' weights (Transfer Learning)
# This is the 'Brain' that knows how to recognize objects and textures
visual_detector = InceptionResNetV2(weights='imagenet')

print("✅ Visual Detection Model (InceptionResNetV2) loaded and ready.")

def analyze_frame(img_path):
    # Prepare the image for the AI (it must be exactly 299x299 pixels)
    img = image.load_img(img_path, target_size=(299, 299))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    
    # Get the prediction
    preds = visual_detector.predict(x)
    return preds

In [ ]:
from transformers import pipeline

# Load the RoBERTa model specifically trained for Social Media sentiment
# Label 0: Negative, Label 1: Neutral, Label 2: Positive
text_integrity_checker = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

def check_text(caption):
    result = text_integrity_checker(caption)
    print(f"Text Analysis Result: {result}")
    return result

# Example: check_text("WARNING: This video of the Minister is 100% REAL and SHOCKING!!")

In [ ]:
import os

# 1. Provide your video name
video_file = 'project.mp4'

# 2. Extract Frames
print("--- Starting Video Processing ---")
extract_frames(video_file)

# 3. Check if frames were created
frame_to_check = 'project_frames/frame_0.jpg'

if os.path.exists(frame_to_check):
    print(f"\n--- Analyzing Frame: {frame_to_check} ---")
    # This calls the visual AI you loaded earlier
    visual_prediction = analyze_frame(frame_to_check)
    print("✅ Visual analysis complete.")
    
    print("\n--- Analyzing Caption Integrity ---")
    # This calls the text AI you just finished loading
    test_caption = "SHOCKING: This leaked footage of the minister is 100% REAL!"
    text_prediction = check_text(test_caption)
    print("✅ Text integrity analysis complete.")
    
    print("\n🚀 PHASE 1 COMPLETE: Your Deepfake Detection Pipeline is fully functional!")
else:
    print(f"❌ Error: Could not find {frame_to_check}. Check if the video file name is correct.")

# 4. Test a sample caption (Module 3)
sample_caption = "You won't believe what happened in this leaked video!"
print("Analyzing text integrity...")
text_results = check_text(sample_caption)

# PASTE THE NEW LINES HERE:
conf_percentage = text_results[0]['score'] * 100
print(f"Confidence Level: {conf_percentage:.2f}%")

print("\n--- PHASE 1 TESTING COMPLETE ---")

In [ ]:
def final_verdict(text_score):
    print("\n--- FINAL SYSTEM VERDICT ---")
    
    # We use the text confidence score as the primary risk indicator for Phase 1
    conf = text_score * 100
    
    if text_score > 0.50:
        print(f"RESULT: ⚠️ SUSPICIOUS CONTENT (Confidence: {conf:.2f}%)")
        print("REASON: High risk detected in semantic integrity and metadata sentiment.")
    else:
        print(f"RESULT: ✅ AUTHENTIC CONTENT (Confidence: {conf:.2f}%)")
        print("REASON: Metadata and visual flow appear within normal parameters.")

# Check if text_results exists, then run
if 'text_results' in locals():
    final_verdict(text_results[0]['score'])
else:
    print("❌ Error: Please run the analysis cell above first!")

In [ ]:
#Phase 2
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# THE CONNECTION: Path to your dataset on F drive
DATA_PATH = r'F:\Deepfake_Project\Celeb_V2\Test'

# Configure the Data Loader (80/20 Train-Validation Split)
datagen = ImageDataGenerator(
    rescale=1./255, 
    validation_split=0.2, 
    horizontal_flip=True, 
    rotation_range=20
)

# Training Set (80% of images)
train_generator = datagen.flow_from_directory(
    DATA_PATH, 
    target_size=(299, 299), 
    batch_size=32, 
    class_mode='binary', 
    subset='training'
)

# Validation Set (20% of images)
val_generator = datagen.flow_from_directory(
    DATA_PATH, 
    target_size=(299, 299), 
    batch_size=32, 
    class_mode='binary', 
    subset='validation'
)

print(f"✅ SUCCESS: Linked to F Drive. Found {train_generator.samples} training images.")

In [ ]:
import sys
!{sys.executable} -m pip install scipy

In [ ]:
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras import layers, models

# Load the base knowledge from Phase 1
base_model = InceptionResNetV2(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False 

# Add the specialized forgery detection layers
model_v2 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5), # Prevents memorizing
    layers.Dense(1, activation='sigmoid') # Output: 0 for Real, 1 for Fake
])

model_v2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("✅ Phase 2 Model Architecture Ready!")

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

# 1. THE SAVE LOCATION (F Drive)
# This will save the "Brain" of your AI to your external drive
checkpoint_path = r'F:\Deepfake_Project\best_deepfake_model.keras'

checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,   # Only updates the file if the AI gets smarter
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

# 2. RUN THE TRAINING
# This uses the 8,000+ images we found in the Celeb_V2\Test folder
print("--- Starting Phase 2 Training ---")
history = model_v2.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[checkpoint_callback]
)

# 3. FINAL BACKUP
model_v2.save(r'F:\Deepfake_Project\final_model_backup.keras')
print("✅ TRAINING COMPLETE: Best model is saved to F:\Deepfake_Project")

In [ ]:
# Use the correct name 'model_v2'
model_v2.save('deepfake_detection_model.h5')
print("✅ Final Model Saved to Disk!")

In [2]:
from tensorflow.keras.models import load_model
model_v2 = load_model('deepfake_detection_model.h5')
print("✅ Model loaded! Ready for detection.")

✅ Model loaded! Ready for detection.


In [ ]:
import sys
!{sys.executable} -m pip install ipywidgets

In [3]:
# RUN THIS CELL TO UPDATE THE DETECTION ENGINE
import cv2
import numpy as np

def predict_video(video_path):
    cap = cv2.VideoCapture(video_path)
    predictions = []
    
    print(f"🔍 HIGH-SENSITIVITY ANALYSIS: {video_path}")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        # INCREASED SENSITIVITY: Scan every 5th frame for better detail
        if int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % 5 == 0:
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # Use INTER_CUBIC to keep the "fake" artifacts sharp
            img = cv2.resize(img, (299, 299), interpolation=cv2.INTER_CUBIC)
            img = img / 255.0
            
            # Predict using the Phase 2 model
            pred = model_v2.predict(np.expand_dims(img, axis=0), verbose=0)[0][0]
            predictions.append(pred)
            
    cap.release()
    
    if predictions:
        avg_score = np.mean(predictions)
        # TWEAK: Deepfakes often have scores around 0.45. 
        # Lowering this threshold from 0.5 to 0.4 catches them better.
        verdict = "⚠️ AI GENERATED (FAKE)" if avg_score > 0.4 else "✅ AUTHENTIC (REAL)"
        confidence = avg_score if avg_score > 0.5 else (1 - avg_score)
        
        print(f"\n{'='*40}")
        print(f"FINAL VERDICT: {verdict}")
        print(f"RAW SCORE: {avg_score:.4f} (Closer to 1.0 = More Fake)")
        print(f"CONFIDENCE: {confidence*100:.2f}%")
        print(f"{'='*40}")

In [8]:
import os
import cv2
import numpy as np

# 1. CHANGE THIS to match the exact name of the file you moved into the folder
manual_video_name = 'demo2.mp4' 

def manual_predict(video_path):
    if not os.path.exists(video_path):
        print(f"❌ Error: {video_path} not found in folder. Check the name!")
        return

    cap = cv2.VideoCapture(video_path)
    predictions = []
    print(f"🔍 Analyzing Manual Upload: {video_path}...")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        # Scan every 5th frame for maximum detail
        if int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % 5 == 0:
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (299, 299), interpolation=cv2.INTER_CUBIC)
            img = img / 255.0
            pred = model_v2.predict(np.expand_dims(img, axis=0), verbose=0)[0][0]
            predictions.append(pred)
    cap.release()
    
    if predictions:
        avg_score = np.mean(predictions)
        # SENSITIVITY FIX: Lowered to 0.3 because compressed videos hide fake details
        verdict = "⚠️ AI GENERATED (FAKE)" if avg_score > 0.3 else "✅ AUTHENTIC (REAL)"
        
        print(f"\n{'='*40}")
        print(f"RESULT: {verdict}")
        print(f"RAW AI SCORE: {avg_score:.4f}")
        print(f"Confidence: {max(avg_score, 1-avg_score)*100:.2f}%")
        print(f"{'='*40}")

# Run it
manual_predict(manual_video_name)

🔍 Analyzing Manual Upload: demo2.mp4...

RESULT: ⚠️ AI GENERATED (FAKE)
RAW AI SCORE: 0.4580
Confidence: 54.20%
